# Week 2 &mdash; Learning on Graphs

## Implementation: Adjacency Matrices, the Laplacian, and the Graph Fourier Transform

This notebook makes the spectral theory of Week 2 concrete using **NetworkX**, **NumPy**, and **SciPy**. We build graphs, compute their Laplacians, examine the eigenstructure, and implement spectral filtering from scratch.

### Setup


In [ ]:
# !pip install numpy scipy networkx matplotlib

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(7)


## 1. Constructing a Graph and Its Matrices

We use a small, interpretable graph: two loosely connected communities. This structure will make the spectral properties (especially the *Fiedler vector*) visually meaningful.


In [ ]:
# Two triangles joined by a single bridge edge -> two communities.
G = nx.Graph()
G.add_edges_from([
    (0, 1), (1, 2), (2, 0),      # community A
    (3, 4), (4, 5), (5, 3),      # community B
    (2, 3),                      # bridge
])

A = nx.to_numpy_array(G)
D = np.diag(A.sum(axis=1))
L = D - A

print("Adjacency A:\n", A)
print("\nDegree diagonal:", np.diag(D))
print("\nLaplacian L = D - A:\n", L)


In [ ]:
pos = nx.spring_layout(G, seed=1)
plt.figure(figsize=(5, 4))
nx.draw(G, pos, with_labels=True, node_color="#8ecae6",
        node_size=700, edge_color="#555")
plt.title("Two communities joined by a bridge")
plt.show()


## 2. Properties of the Laplacian

We verify the theoretical claims from the notebook: positive semi-definiteness, the constant vector in the kernel, and the connection between the smallest eigenvalues and graph connectivity.


In [ ]:
evals, evecs = np.linalg.eigh(L)   # eigh: symmetric -> real, sorted ascending
print("Eigenvalues of L:", np.round(evals, 4))

# 1. Smallest eigenvalue is 0 with the constant eigenvector.
print("\nlambda_1 ~ 0 ?", np.isclose(evals[0], 0))
print("u_1 constant ?", np.allclose(np.abs(evecs[:, 0]), np.abs(evecs[0, 0])))

# 2. PSD: x^T L x >= 0 and equals the Dirichlet energy.
x = rng.normal(size=L.shape[0])
quad = x @ L @ x
dirichlet = 0.5 * sum(A[i, j] * (x[i] - x[j])**2
                      for i in range(len(x)) for j in range(len(x)))
print("\nx^T L x            =", round(quad, 4))
print("Dirichlet energy   =", round(dirichlet, 4))


### The Fiedler vector

The eigenvector $u_2$ associated with the *second-smallest* eigenvalue $\lambda_2$ (the **algebraic connectivity**) is the **Fiedler vector**. Its sign pattern reveals the community structure &mdash; the foundation of spectral clustering.


In [ ]:
fiedler = evecs[:, 1]
print("Fiedler vector:", np.round(fiedler, 3))

colors = ["#e76f51" if v > 0 else "#2a9d8f" for v in fiedler]
plt.figure(figsize=(5, 4))
nx.draw(G, pos, with_labels=True, node_color=colors, node_size=700, edge_color="#555")
plt.title("Communities recovered from the sign of the Fiedler vector")
plt.show()


The two communities are perfectly separated by the sign of the Fiedler vector &mdash; a striking demonstration that the *spectrum encodes graph geometry*.


## 3. The Graph Fourier Transform

We define the GFT via the Laplacian eigenbasis and confirm it is an orthogonal change of basis: signals can be analysed (projected onto frequencies) and exactly synthesised back.


In [ ]:
def gft(x, U):
    return U.T @ x        # analysis: signal -> spectral coefficients

def igft(xhat, U):
    return U @ xhat       # synthesis: coefficients -> signal

U = evecs
# A smooth signal (varies slowly across the graph) vs a noisy one.
smooth = fiedler.copy()
noisy = rng.normal(size=len(smooth))

print("GFT of smooth signal (energy concentrated at low frequency):")
print(np.round(gft(smooth, U), 3))
print("\nGFT of noisy signal (energy spread across frequencies):")
print(np.round(gft(noisy, U), 3))

# Round-trip exactness.
print("\nReconstruction error:", np.abs(igft(gft(noisy, U), U) - noisy).max())


## 4. Spectral Filtering from Scratch

We implement a **low-pass filter** $g(\lambda) = 1/(1 + \alpha\lambda)$ in the spectral domain and use it to denoise a graph signal. This is the explicit $U g(\Lambda) U^\top$ construction of the theory notebook.


In [ ]:
def spectral_filter(x, U, evals, g):
    """Apply filter g(.) defined on eigenvalues: U g(Lambda) U^T x."""
    xhat = U.T @ x
    return U @ (g(evals) * xhat)

# Ground-truth smooth signal + noise.
clean = fiedler.copy()
observed = clean + 0.4 * rng.normal(size=len(clean))

low_pass = lambda lam, alpha=5.0: 1.0 / (1.0 + alpha * lam)
denoised = spectral_filter(observed, U, evals, low_pass)

print("Error before filtering:", np.round(np.abs(observed - clean), 3))
print("\n||observed - clean|| =", round(np.linalg.norm(observed - clean), 4))
print("||denoised - clean|| =", round(np.linalg.norm(denoised - clean), 4))


The low-pass filter attenuates high-frequency Laplacian modes &mdash; exactly the components dominated by noise &mdash; reducing the reconstruction error. This is the spectral-domain mechanism that learnable graph convolutions exploit.


## 5. Polynomial Filters and Locality

Finally we verify the locality claim: a degree-$K$ polynomial of $L$ only connects vertices within $K$ hops. We compare $L^k$ sparsity patterns to graph distances.


In [ ]:
def khop_support(L, k):
    M = np.linalg.matrix_power(L, k)
    return (np.abs(M) > 1e-9).astype(int)

dist = dict(nx.all_pairs_shortest_path_length(G))
n = G.number_of_nodes()
Dmat = np.array([[dist[i][j] for j in range(n)] for i in range(n)])

for k in [1, 2, 3]:
    supp = khop_support(L, k)
    # Every nonzero of L^k must correspond to graph distance <= k.
    ok = np.all((supp == 1) <= (Dmat <= k))
    print(f"L^{k}: support contained in {k}-hop neighbourhood? {bool(ok)}")


In [ ]:
# Visualise the growing support of L^k.
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, k in zip(axes, [1, 2, 3]):
    ax.imshow(khop_support(L, k), cmap="Blues")
    ax.set_title(f"Support of $L^{k}$")
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
plt.suptitle("Polynomial filters of the Laplacian are K-localised")
plt.tight_layout()
plt.show()


## 6. Summary and Exercises

- We built graphs and their $A$, $D$, $L$ matrices with NetworkX.
- We verified the Laplacian's PSD property, its kernel, and the **Fiedler vector**'s community-detection power.
- We implemented the **graph Fourier transform** and a **spectral low-pass filter** for denoising.
- We confirmed that **polynomial filters** $L^k$ are exactly $K$-localised, motivating the spatial GNNs of Week 3.

### Exercises

1. Repeat the Fiedler analysis on a graph with *three* communities. How many near-zero eigenvalues appear?
2. Implement a **high-pass** filter $g(\lambda) = \alpha\lambda/(1+\alpha\lambda)$ and visualise its effect on a smooth signal.
3. Compare the normalised Laplacian $L_{\mathrm{sym}} = D^{-1/2}LD^{-1/2}$ spectrum (bounded in $[0,2]$) with that of $L$.
